# Long term

In [57]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact, widgets
from IPython.display import display

In [58]:
%load_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [136]:
#Run this to reload the python file
%autoreload 2
from utils import *

## Import data

In [60]:
# read the file with the pcp information from GPM and IMN
imn_df = pd.read_csv('./Data/harmonized/unif_IMN.csv')
gpm_df = pd.read_csv('./Data/harmonized/unif_GPM.csv')

In [ ]:
#meta = pd.read_csv('./Data/metadata/IMN_stations_rev.csv')

In [ ]:
#gpm_coord = pd.read_csv('./Data/metadata/gpm_coords.csv')

In [61]:
meta_df = pd.read_csv('./Data/metadata/long_term_v07.csv')

## Pre process

In [62]:
# check the len of the dfs
print(len(imn_df))
print(len(gpm_df))

192840
192840


In [63]:
# assigning 'date' as index
imn_df = convert_index(imn_df, 'date')
gpm_df = convert_index(gpm_df, 'date')

In [64]:
# match gpm_df dates with imn_df
imn_df = imn_df['2001-01-01 00:00:00':'2022-12-31 23:00:00']
gpm_df = gpm_df['2001-01-01 00:00:00':'2022-12-31 23:00:00']

In [65]:
temp_resol = {
    '1D': '1 día',
    '7D': '1 semana',
    '1M': '1 mes'
}

In [ ]:
#merged_df = pd.merge(meta, gpm_coord, left_on='Número', right_on='number')

In [ ]:
#merged_df = merged_df.drop(columns=['Unnamed: 0.1_x', 'Unnamed: 0_x', 'station_number', 'Unnamed: 0.1_y', 'Unnamed: 0_y', 'number'])

## Analysis

### Correlations

In [ ]:
corr_temp = []
for window in temp_resol.keys():
    tmp = corr_accum(imn_df, gpm_df, window)
    result_dict = {
        'window': window,
        'correlation': tmp.get('correlation')
    }
    corr_temp.append(result_dict)

In [ ]:
df = pd.DataFrame()
df['1D'] = corr_temp[0]['correlation']
df['7D'] = corr_temp[1]['correlation']
df['1M'] = corr_temp[2]['correlation']

In [ ]:
df = df.reset_index()
df.rename(columns={'index': 'numero'}, inplace=True)

In [ ]:
df['numero'] = df['numero'].astype(float)

In [ ]:
merged_df = pd.merge(merged_df, df, left_on='Número', right_on='numero')
merged_df = merged_df.drop(columns=['numero'])

### Distance between points

In [ ]:
#gpm_coord['distance'] = gpm_coord.apply(lambda row: haversine(row['lat'],
#                                                              row['lon'],
#                                                              meta.loc[meta['station_number'] == row['number'], 'lat'].iloc[0],
#                                                              meta.loc[meta['station_number'] == row['number'], 'lon'].iloc[0]),
#                                        axis=1)

In [ ]:
#gpm_coord.to_csv('./Data/metadata/gpm_coords.csv')

In [ ]:
#tmp = meta.merge(gpm_coord, left_on='Número', right_on='number')
#tmp = tmp.merge(df, left_on='Número', right_on='numero')
#tmp = tmp.drop(columns=['Unnamed: 0.1_x', 'Unnamed: 0_x', 'station_number', 'Unnamed: 0.1_y','Unnamed: 0_y', 'number', 'numero'])
#tmp.to_csv('./Data/metadata/long_term_v07.csv')

In [ ]:
meta_df = pd.read_csv('./Data/metadata/long_term_v07.csv')

### Differences between accumulated

In [10]:
# List to store dictionaries
result_list = []

# Loop through columns and append results to the list
for number in imn_df.columns:
    result_dict = diff_accum(imn_df, gpm_df, number)
    result_list.append(result_dict)

# Convert the list of dictionaries into a DataFrame
accumulated_diff = pd.DataFrame(result_list)

In [15]:
accumulated_diff['number'] = accumulated_diff['number'].astype(float)

In [16]:
merged_df = meta_df.merge(accumulated_diff, left_on='Número', right_on='number')

In [18]:
merged_df = merged_df.drop(columns='number')

In [19]:
#merged_df.to_csv('./Data/metadata/long_term_v07.csv')

### Asignación de Zonas Climáticas

In [ ]:
zonas_climaticas = {
    'Pacifico Norte': {'74051', '74053', '76055', '72157'},
    'Pacifico Sur': {'98091', '98087', '98095', '98075'},
    'Zona Norte': {'69679', '69633'},
    'Valle Central': {'84139', '84141', '84169', '84187', '84191', '84197', '73123'},
    'Vertiente del Caribe': {'81005', '69681', '71015'}
}

In [ ]:
merged_df['region_climatica'] = None

In [ ]:
# Assign the corresponding climate zone based on the zonas_climaticas diccionary
for index, row in merged_df.iterrows():
    number = str(row['Número'])

    for zone, station_numbers in zonas_climaticas.items():
        if number in station_numbers:
            merged_df.at[index, 'region_climatica'] = zone
            break  

In [ ]:
merged_df = merged_df.sort_values(by='region_climatica')

In [ ]:
merged_df.to_csv('./Data/metadata/long_term_v07.csv')

## Plotting

### Correlations

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Bar width can be adjusted based on preference
bar_width = 0.2
bar_positions = np.arange(len(meta_df['Número'].unique()))

# Define colors for each bar
color_1d = '#98df8a'  # Light green
color_7d = '#2ca02c'  # Green
color_1m = '#1f771f'  # Dark green

# Plot each column as a separate bar with assigned colors
ax.bar(bar_positions - bar_width, meta_df['1D'], width=bar_width, label='1 día', color=color_1d)
ax.bar(bar_positions, meta_df['7D'], width=bar_width, label='1 semana', color=color_7d)
ax.bar(bar_positions + bar_width, meta_df['1M'], width=bar_width, label='1 mes', color=color_1m)

# Set labels and title
ax.set_xlabel('Número de estación')
ax.set_ylabel('Correlación')
ax.set_title('Correlaciones según la EMA y el acumulado temporal')
ax.legend()

plt.xticks(bar_positions, meta_df['Número'].unique())
plt.xticks(rotation=45)

plt.tight_layout()

# Show the plot
plt.show()

### Altitude vs Temporal Accumulated Correlation 

In [ ]:
# Create a dropdown widget for selecting the column
column_options = list(temp_resol.keys())
column_dropdown = widgets.Dropdown(
    options=column_options,
    value=column_options[0],
    description='Acumulado:',
    disabled=False
)

In [ ]:
# Create an interactive plot
interactive_plot = widgets.interactive(
    scatter_plot_log,
    df=widgets.fixed(meta_df),
    column=column_dropdown
)

# Display the interactive plot
display(interactive_plot)

### Location vs Difference in the accumulated

In [ ]:
plt.figure(figsize=(10, 6))
# Define color map based on unique values in 'region_climatica'
colors = {'Pacifico Norte': 'b', 
          'Pacifico Sur': 'r', 
          'Valle Central': 'g', 
          'Vertiente del Caribe':'c',
          'Zona Norte':'m' 
          }
    
# Iterate through each region to create scatter plots with appropriate labels
for region, color in colors.items():
    region_data = meta_df[meta_df['region_climatica'] == region]
    plt.scatter((-1)*region_data['percentage_diff'], region_data['Altitud (m.s.n.m.)'], c=color, label=region)

plt.xlabel('[%]')
plt.ylabel('[m.s.n.m] (escala log)')
plt.title(f'Altitud de EMA vs Diferencia Porcentual')
plt.legend()
plt.yscale('log') 
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
# Define color map based on unique values in 'region_climatica'
colors = {'Pacifico Norte': 'b', 
          'Pacifico Sur': 'r', 
          'Valle Central': 'g', 
          'Vertiente del Caribe':'c',
          'Zona Norte':'m' 
          }
    
# Iterate through each region to create scatter plots with appropriate labels
for region, color in colors.items():
    region_data = meta_df[meta_df['region_climatica'] == region]
    plt.scatter((-1)*region_data['diff_accum'], region_data['Altitud (m.s.n.m.)'], c=color, label=region)

plt.xlabel('[mm]')
plt.ylabel('[m.s.n.m] (escala log)')
plt.title(f'Altitud de EMA vs Diferencia')
plt.legend()
plt.yscale('log') 
plt.show()

### Time Series

In [66]:
# Create a dropdown widget for selecting the column
column_options = list(imn_df.columns)  # Assuming both dataframes have the same columns
column_dropdown = widgets.Dropdown(
    options=column_options,
    value=column_options[0],
    description='Station Number:',
    disabled=False
)

In [14]:
# Create a dropdown widget for selecting resolution
resolution_options = list(temp_resol.keys()) 
resolution_dropdown = widgets.Dropdown(
    options=resolution_options,
    value=resolution_options[0],
    description='Resolution:',
    disabled=False
)

In [15]:
# Create an interactive plot
interactive_plot = widgets.interactive(
    plot_resol,
    df1=widgets.fixed(imn_df),
    df2=widgets.fixed(gpm_df),
    resolution=resolution_dropdown,
    column=column_dropdown
)

# Display the interactive plot
display(interactive_plot)

interactive(children=(Dropdown(description='Resolution:', options=('1D', '7D', '1M'), value='1D'), Dropdown(de…

### Frequency Spectrum

In [67]:
imn_tmp = imn_df.fillna(0)

In [137]:
# Create an interactive plot
interactive_plot = widgets.interactive(
    plot_freq_2,
    df1=widgets.fixed(imn_tmp),
    df2=widgets.fixed(gpm_df),
    column=column_dropdown
)
# Display the interactive plot
display(interactive_plot)

interactive(children=(Dropdown(description='Station Number:', index=19, options=('84141', '69679', '74051', '7…

In [103]:
# Find the top 5 highest signals from Fourier Transform 
d = 3600  # for hourly data

# Find top signals for IMN data
top_signals_IMN = find_top_signals(imn_tmp, d)

# Find top signals for GPM data
top_signals_GPM = find_top_signals(gpm_df, d)

In [104]:
# Apply the transformation from frequency to time
for col in top_signals_IMN.columns:
    if 'frequency' in col:
        top_signals_IMN[col] = top_signals_IMN[col].apply(lambda x: 1 / x * (1 / 86400)).round(3)

for col in top_signals_GPM.columns:
    if 'frequency' in col:
        top_signals_GPM[col] = top_signals_GPM[col].apply(lambda x: 1 / x * (1 / 86400)).round(3)

In [105]:
top_signals_GPM

,frequency_1,frequency_2,frequency_3,frequency_4,frequency_5,frequency_6,frequency_7,frequency_8,frequency_9,frequency_10,magnitude_1,magnitude_2,magnitude_3,magnitude_4,magnitude_5,magnitude_6,magnitude_7,magnitude_8,magnitude_9,magnitude_10
71015,365.227,182.614,349.348,121.742,186.860,0.500,1.008,382.619,7.885,46.445,21106.785712,16945.024773,11234.560148,9772.376038,8634.904032,7010.180448,6984.151999,6696.327778,6647.876476,6476.017689
73123,365.227,1.000,0.250,1.000,2.286,0.640,0.500,1.003,0.281,0.390,18319.702026,16419.537955,12847.470015,12363.509372,12189.767698,11571.026900,10684.166581,10535.413035,10258.883137,9403.093882
69633,365.227,1.000,382.619,1.000,349.348,1.003,0.997,1.003,121.742,1.000,23951.138699,16199.276466,10513.888549,9498.231038,8894.220863,8754.404046,7649.963873,6502.089726,5959.720532,5867.535541
74051,365.227,1.000,0.997,0.500,1.003,121.742,382.619,1.000,1.000,349.348,24147.299074,23557.920169,14109.815660,12685.004834,12301.872543,10950.752497,10807.927389,10457.919281,10142.035325,9987.578082
98087,1.000,0.500,365.227,1.000,1.000,1.003,0.997,182.614,0.333,0.500,44044.602666,23738.239316,22623.092697,19817.207681,17498.719475,15524.769999,15146.962100,13901.840987,11294.926804,10065.808856
81005,365.227,182.614,1.000,349.348,9.332,1.000,22.698,186.860,22.319,16.135,15661.740926,12570.351281,10189.884787,9051.768502,6870.908305,6813.888607,6623.834728,6619.939874,6304.605785,6243.127936
98091,1.000,365.227,0.500,1.000,1.000,182.614,1.003,0.997,0.500,382.619,34115.775072,20972.740982,15754.999862,14962.481986,14691.668222,12765.073228,12572.701203,10893.286429,9354.155459,9105.899164
84139,0.500,0.500,365.227,0.250,8035.000,0.490,1.003,0.490,0.250,382.619,17401.024240,17365.708587,16046.842148,11100.760058,9539.896159,9260.695249,8459.697245,8333.477469,7741.861998,7399.988401
98075,1.000,365.227,0.500,1.000,1.000,1.003,0.997,182.614,0.500,349.348,46867.714060,23348.425561,23250.260626,20642.349626,19936.265986,15490.421882,14230.089616,13605.221447,10684.147761,10230.509906
84191,365.227,1.000,0.500,0.500,382.619,0.997,1.003,349.348,1.000,1.000,24873.675180,22922.778815,17826.597022,12237.708334,11693.371161,11329.226451,11094.147534,10883.907458,9434.776667,8457.588732
